## Overview

This notebook presents the most significant experiments from our Food Hazard Detection system.

## Experimental Pipeline

| # | Method | Kaggle Score | Key Insight | Notebook |
|---|--------|-------------|-------------|----------|
| 1 | TF-IDF + SVM (tuned) | 0.741 | Strong classical baseline | food_hazard_phase1 |
| 2 | DistilBERT + SVM Ensemble | 0.755 | First transformer ensemble | phase4_ensemble_bert_svm |
| 3 | Multi-task BERT | 0.771 | Shared encoder for both tasks | phase20_multitask_bert |
| 4 | BERT-base + Focal Loss | 0.804 | Key breakthrough: imbalance handling | phase15_bertbase_focal |
| 5 | BERT + DistilBERT (0.8/0.2) | 0.806 | Ensemble exploration | phase16_bertbase_distil_ensemble |
| 6 | BERT + Multi-task (0.5/0.5) | 0.817 | Heterogeneous ensemble | phase21_three_model_ensemble |
| 7 | BERT haz + Ensemble product | **0.819** | **Best submission** | phase26_mixed_classifiers |
| 8 | BERT-large + BERT-base(best) + Multi-task | 0.790 | **Best private score submission: 0.85010** | phase41_bert_large_multilingual | 
| 9 | T5 Text-to-Text Classification | 0.654 | Text-to-text generation adds inference overhead without performance gain | phase36_t5 |
| 10 | EDA + Targeted Augmentation | 0.770 | WordNet synonyms corrupt domain-specific terms — augmentation hurts | phase37_eda_augmentation |
| extra | analysis, comparisons, etc | --- | --- | phase23_analysis, phase30_chi2_feature_selection, phase_final_comparison |

# 1) TF-IDF + SVM (tuned)  
*The method will only run in its respective notebook!!*

In [ ]:
# Tuning της παραμέτρου C για το SVM
C_values = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
results = []

for C in C_values:
    # Hazard classifier
    clf_h = LinearSVC(C=C, class_weight='balanced', max_iter=2000, random_state=42)
    clf_h.fit(X_train_best, y_train_hazard)
    
    # Product classifier
    clf_p = LinearSVC(C=C, class_weight='balanced', max_iter=2000, random_state=42)
    clf_p.fit(X_train_best, y_train_product)
    
    # Αξιολόγηση
    pred_h = clf_h.predict(X_valid_best)
    pred_p = clf_p.predict(X_valid_best)
    
    score = official_st1_score(
        valid['hazard-category'], pred_h,
        valid['product-category'], pred_p,
        verbose=False  # δεν εκτυπώνουμε αναλυτικά για κάθε C
    )
    
    results.append({'C': C, 'score': score})
    print(f'C={C:.2f} → Score: {score:.4f}')

# Εύρεση βέλτιστου C
best = max(results, key=lambda x: x['score'])
print(f'\nΒέλτιστο C: {best["C"]} με score: {best["score"]:.4f}')
print(f'Προηγούμενο best (C=1.0): 0.7263')

In [ ]:
# Tuning max_features στο TF-IDF
feature_values = [5000, 10000, 20000, 50000, 100000]
results_features = []

for max_feat in feature_values:
    # Νέο TF-IDF
    tfidf_tune = TfidfVectorizer(
        max_features=max_feat,
        ngram_range=(1, 2),
        sublinear_tf=True,
        stop_words='english'
    )
    
    X_tr = tfidf_tune.fit_transform(train['combined'])
    X_vl = tfidf_tune.transform(valid['combined'])
    
    # SVM με βέλτιστο C=0.5
    clf_h = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
    clf_h.fit(X_tr, y_train_hazard)
    
    clf_p = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
    clf_p.fit(X_tr, y_train_product)
    
    pred_h = clf_h.predict(X_vl)
    pred_p = clf_p.predict(X_vl)
    
    score = official_st1_score(
        valid['hazard-category'], pred_h,
        valid['product-category'], pred_p,
        verbose=False
    )
    
    results_features.append({'max_features': max_feat, 'score': score})
    print(f'max_features={max_feat} → Score: {score:.4f}')

best_feat = max(results_features, key=lambda x: x['score'])
print(f'\nΒέλτιστο max_features: {best_feat["max_features"]} με score: {best_feat["score"]:.4f}')
print(f'Προηγούμενο best (10000): 0.7388')

In [ ]:
# Εκπαίδευση με βέλτιστες παραμέτρους: C=0.5, max_features=50000
tfidf_final = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)

X_tr_final = tfidf_final.fit_transform(train['combined'])
X_vl_final = tfidf_final.transform(valid['combined'])
X_te_final = tfidf_final.transform(test['combined'])

clf_h_final = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_h_final.fit(X_tr_final, y_train_hazard)

clf_p_final = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_p_final.fit(X_tr_final, y_train_product)

# Επαλήθευση στο validation
pred_h_final = clf_h_final.predict(X_vl_final)
pred_p_final = clf_p_final.predict(X_vl_final)

print('=== ΤΕΛΙΚΗ ΑΞΙΟΛΟΓΗΣΗ ===\n')
official_st1_score(
    valid['hazard-category'], pred_h_final,
    valid['product-category'], pred_p_final
)

# Παραγωγή submission
pred_h_test = clf_h_final.predict(X_te_final)
pred_p_test = clf_p_final.predict(X_te_final)

submission_final = pd.DataFrame({
    'id': test['id'],
    'hazard-category': pred_h_test,
    'product-category': pred_p_test
})

submission_final.to_csv('submission_tuned.csv', index=False)
print('\nSubmission file saved: submission_tuned.csv')
print(f'Αριθμός predictions: {len(submission_final)}')